# Phase 6 Telegram Bot 測試

1. 複製 `.env.example` 為專案根目錄 `.env` 並填入 `TELEGRAM_BOT_TOKEN`、`TELEGRAM_CHAT_ID`
2. 執行下方儲存格做離線報告格式測試（不需連線 Telegram）

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from bot.model_predictor import calculate_recommended_strike, get_full_prediction
from bot.notification_formatter import format_telegram_message

# 離線：履約價單元測試
strike = calculate_recommended_strike(
    symbol="BTC",
    current_price=100_000,
    daily_realized_vol=0.02,
    direction="bullish",
    direction_conf=0.65,
    kurtosis_high=False,
)
assert strike["put_strike"] < 100_000
print("strike OK:", strike["put_strike"], strike["call_strike"])

# 離線：完整預測 + HTML 格式化（不送 Telegram）
from bot.notification_scheduler import build_fallback_report

try:
    data = get_full_prediction()
except Exception as exc:
    data = build_fallback_report(str(exc))
    print("（資料過期或異常，使用 fallback dict）", exc)
msg = format_telegram_message(data)
print(msg[:1200], "\n... [truncated for notebook display]")

In [ ]:
# 需設定 TELEGRAM_BOT_TOKEN + TELEGRAM_CHAT_ID 後取消註解：
# !python ../run_telegram_bot.py --test